# 2.1 - LBP + SVM (pré-processamento v2: MTCNN 224, margem 56)

Mesma estrutura do `2.0`: uma célula por experimento (você roda um de cada vez), comparação na validação,
e o teste numa célula própria, com `BEST_EXP_ID` escolhido por você.

Diferenças em relação ao `2.0`:
- Lê as imagens v2 (`TRAIN_V2_DIR`, `VAL_V2_DIR`, `TEST_V2_DIR`) e usa `src/features_v2.py`:
  LBP uniforme u2 (59 e 243 bins), rosto reduzido para 64x64 dentro da extração, Maatta com regiões de 14 px de sobreposição.
  Dimensões: global 59, grid 531, Maatta 833.
- Tudo é salvo só em `TCC/v2/` (características, log). O `experiment_log.csv` e a pasta `experiments/lbp_svm` anteriores não são tocados.
- Log próprio `TCC/v2/experiment_log_v2.csv`, com as colunas do log anterior mais EER.
- As características de cada experimento ficam salvas em disco. **Se o Colab desconectar**, rode as células 1 a 6 e depois só o que faltou:
  o que já foi extraído é carregado do disco, e a comparação lê o log.
- O teste: o modelo é retreinado a partir das características salvas, então não depende de variáveis na memória.

**Pré-requisito:** copiar `features_v2.py` para `src/` no repositório e subir para o GitHub.


In [ ]:
# Clona o repositório (ou reaproveita, se já existir) e instala as dependências.
# facenet-pytorch precisa de --no-deps (ver 2.0). Este notebook não roda MTCNN,
# mas src/dataset.py importa o facenet-pytorch.
import os
if not os.path.isdir('/content/liveness-detection'):
    !git clone https://github.com/laianemuckler/liveness-detection.git
%cd /content/liveness-detection
!git pull --quiet

!pip install -r requirements.txt --quiet
!pip install facenet-pytorch==2.6.0 --no-deps --quiet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import csv
import json
from datetime import date, datetime

import numpy as np
import pandas as pd
from PIL import Image

from src.config import (
    TRAIN_V2_DIR, VAL_V2_DIR, TEST_V2_DIR,
    DRIVE_ROOT, LABEL_LIVE, LABEL_SPOOF,
)
from src.dataset import load_processed_split
from src.features_v2 import extract_features_v2, EXPECTED_DIMS_V2
from src.modeling.train import train_svm
from src.modeling.predict import evaluate

# ---------------- Saídas do v2 (pastas novas, nada do v1 é tocado) ----------------
OUTPUT_V2_ROOT = os.path.join(DRIVE_ROOT, 'v2')
RUNS_DIR = os.path.join(OUTPUT_V2_ROOT, 'experiments', 'lbp_svm', 'runs')
LOG_V2_PATH = os.path.join(OUTPUT_V2_ROOT, 'experiment_log_v2.csv')
os.makedirs(RUNS_DIR, exist_ok=True)

# Contagens do protocolo (live, spoof) e dimensões esperadas de cada configuração (src/features.py)
EXPECTED_COUNTS = {'train': (1407, 1366), 'val': (336, 382), 'test': (3362, 5761)}
EXPECTED_DIMS = EXPECTED_DIMS_V2   # 59, 531 e 833 (LBP uniforme u2 de 59/243 bins, rosto em 64x64)

# id do experimento v2 -> configuração de características
CONFIGS_V2 = {
    'lbp_svm_v2_01_global': 'lbp_u2_global_p8r1',
    'lbp_svm_v2_02_grid': 'lbp_u2_grid_p8r1_3x3',
    'lbp_svm_v2_03_maatta': 'lbp_u2_maatta',
}

In [ ]:
# Log do v2: mesmas colunas e formatação de src.modeling.predict.log_experiment,
# mas gravando em TCC/v2/experiment_log_v2.csv (o log do v1 não é tocado).

def log_experiment_v2(exp_id, metodo, feature_config, modelo_config,
                      hter_val=None, auc_val=None, hter_test=None, auc_test=None,
                      eer_val=None, eer_test=None, obs=''):
    def fmt(v):
        return f"{v:.4f}" if v is not None else ''

    row = {
        'exp_id': exp_id, 'metodo': metodo, 'feature_config': feature_config,
        'modelo_config': modelo_config,
        'HTER_val': fmt(hter_val), 'AUC_val': fmt(auc_val), 'EER_val': fmt(eer_val),
        'HTER_test': fmt(hter_test), 'AUC_test': fmt(auc_test), 'EER_test': fmt(eer_test),
        'data': date.today().isoformat(), 'obs': obs,
    }
    file_exists = os.path.isfile(LOG_V2_PATH)
    os.makedirs(os.path.dirname(LOG_V2_PATH), exist_ok=True)
    with open(LOG_V2_PATH, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)
    print(f"Registrado '{exp_id}' em {LOG_V2_PATH}")


def ja_registrado(exp_id):
    """True se esse exp_id já está no log do v2 (evita duplicar linhas ao rodar de novo)."""
    if not os.path.isfile(LOG_V2_PATH):
        return False
    return exp_id in set(pd.read_csv(LOG_V2_PATH, dtype=str)['exp_id'])

In [ ]:
# Carrega as listas de imagens v2 e confere com o protocolo

train_paths, y_train = load_processed_split(TRAIN_V2_DIR)
val_paths, y_val = load_processed_split(VAL_V2_DIR)
test_paths, y_test = load_processed_split(TEST_V2_DIR)

print(f"Train: {len(train_paths)} | Validation: {len(val_paths)} | Test: {len(test_paths)}")

for nome, y_, esperado in (('train', y_train, EXPECTED_COUNTS['train']),
                           ('val', y_val, EXPECTED_COUNTS['val']),
                           ('test', y_test, EXPECTED_COUNTS['test'])):
    y_ = np.array(y_)
    encontrado = (int(np.sum(y_ == LABEL_LIVE)), int(np.sum(y_ == LABEL_SPOOF)))
    print(f"  {nome:<5} live/spoof = {encontrado}")
    if encontrado != esperado:
        raise ValueError(f"Contagem de '{nome}' não bate com o protocolo: "
                         f"esperado {esperado}, encontrado {encontrado}.")

# garante que são as imagens v2 (224) e não as do v1 (160)
with Image.open(train_paths[0]) as im0:
    if im0.size != (224, 224):
        raise ValueError(f"Imagem com tamanho {im0.size}; o esperado no v2 é (224, 224).")
print("Imagens v2 conferidas (224x224).")

In [ ]:
# Funções do experimento (só definem; não rodam nada pesado)

def get_features(exp_dir, split, paths, feature_config_name):
    """Carrega X_<split>.npy se já existir (e confere o formato); senão extrai e salva.
    Nunca sobrescreve um arquivo existente."""
    npy_path = os.path.join(exp_dir, f'X_{split}.npy')
    esperado = (len(paths), EXPECTED_DIMS[feature_config_name])

    if os.path.exists(npy_path):
        X = np.load(npy_path)
        if X.shape != esperado:
            raise RuntimeError(f"{npy_path} tem formato {X.shape}, esperado {esperado}. "
                               f"Não vou sobrescrever: confira a pasta antes de continuar.")
        print(f"[{os.path.basename(exp_dir)}] X_{split}: carregado do disco {X.shape}")
        return X

    print(f"[{os.path.basename(exp_dir)}] extraindo '{feature_config_name}' ({split})...")
    X = extract_features_v2(paths, feature_config_name)
    if X.shape != esperado:
        raise ValueError(f"Formato inesperado {X.shape}; esperado {esperado}.")
    np.save(npy_path, X)
    return X


def run_lbp_experiment(exp_id, feature_config_name, svm_kernel='rbf', svm_C=1.0):
    """extrair (treino e validação) -> treinar SVM -> avaliar na validação -> registrar no log v2.
    As características do TESTE não são extraídas aqui."""
    exp_dir = os.path.join(RUNS_DIR, exp_id)
    os.makedirs(exp_dir, exist_ok=True)

    X_train = get_features(exp_dir, 'train', train_paths, feature_config_name)
    X_val = get_features(exp_dir, 'val', val_paths, feature_config_name)

    print(f"[{exp_id}] treinando SVM...")
    model, scaler = train_svm(X_train, y_train, kernel=svm_kernel, C=svm_C)
    val_results = evaluate(model, scaler, X_val, y_val)
    print(f"[{exp_id}] Validação -> HTER: {val_results['HTER']*100:.2f}% | "
          f"AUC: {val_results['AUC']:.4f} | EER: {val_results['EER']*100:.2f}%")

    if ja_registrado(exp_id):
        print(f"[{exp_id}] já está no log do v2; não registrei de novo.")
    else:
        log_experiment_v2(
            exp_id=exp_id, metodo='LBP+SVM (v2 224)', feature_config=feature_config_name,
            modelo_config=f"kernel={svm_kernel}, C={svm_C}",
            hter_val=val_results['HTER'], auc_val=val_results['AUC'], eer_val=val_results['EER'],
            obs='pre-processamento v2: MTCNN 224, margem 56',
        )

    return {'feature_config': feature_config_name, 'svm_kernel': svm_kernel,
            'svm_C': svm_C, 'val_results': val_results}

In [ ]:
# Experimento 1: LBP global (u2, P=8, R=1), um histograma para o rosto (59 dimensões)

exp1 = run_lbp_experiment('lbp_svm_v2_01_global', CONFIGS_V2['lbp_svm_v2_01_global'])

# Experimento 2: LBP grid (u2, P=8, R=1, 3x3 blocos)

Features por bloco (per-block), Chingovska et al. (2012). 531 dimensões.

In [ ]:
# Experimento 2: LBP grid (u2, P=8, R=1, 3x3 blocos)

exp2 = run_lbp_experiment('lbp_svm_v2_02_grid', CONFIGS_V2['lbp_svm_v2_02_grid'])

# Experimento 3: LBP Maatta (u2: P=8,R=1 em 3x3 regiões sobrepostas de 14 px + global P=8,R=2 + global P=16,R=2)

Määttä, Hadid e Pietikäinen (2011). 833 dimensões.

In [ ]:
# Experimento 3: LBP Maatta

exp3 = run_lbp_experiment('lbp_svm_v2_03_maatta', CONFIGS_V2['lbp_svm_v2_03_maatta'])

In [ ]:
# Compara os experimentos na validação (lê do log, então funciona mesmo depois de reconectar o Colab)

log = pd.read_csv(LOG_V2_PATH, dtype=str)
val_rows = log[log['HTER_val'].notna() & (log['HTER_val'] != '') & ~log['exp_id'].str.endswith('_FINAL_TEST')]
display(val_rows[['exp_id', 'feature_config', 'HTER_val', 'AUC_val', 'EER_val']])

In [ ]:
# Medição de tempo e hardware. A lógica geral está em src/timing.py (igual para todos os notebooks);
# aqui fica só o que é do LBP (qual função extrai, como treina). Grava em TCC/v2/timing_v2.csv.
# - extração: média por imagem, 1 processo, em uma AMOSTRA da VALIDAÇÃO copiada para o disco local
#   (o tempo não inclui a leitura lenta do Drive e não usa o teste)
# - treino do SVM: train_svm com as características já salvas
# - predição: scaler + SVM por imagem, na validação (o teste é medido na célula do teste)
from src.timing import (medir_s, medir_por_item, copiar_amostra, tem_tempo, gravar_tempo,
                        gravar_tempo_teste, carregar_tempos, hardware_info)
from src.features_v2 import FEATURE_CONFIGS_V2

N_AMOSTRA_TEMPO = 200
TIMING_V2_PATH = os.path.join(OUTPUT_V2_ROOT, 'timing_v2.csv')


def medir_tempo_validacao(exp_id, feature_config_name):
    if tem_tempo(TIMING_V2_PATH, exp_id, 'validacao'):
        print(f"[{exp_id}] já tem tempo medido; não medi de novo.")
        return
    exp_dir = os.path.join(RUNS_DIR, exp_id)
    f_tr, f_va = os.path.join(exp_dir, 'X_train.npy'), os.path.join(exp_dir, 'X_val.npy')
    if not (os.path.exists(f_tr) and os.path.exists(f_va)):
        print(f"[{exp_id}] sem características salvas; rode o experimento antes.")
        return

    amostra = copiar_amostra(val_paths, N_AMOSTRA_TEMPO, '/content/amostra_tempo')
    extracao_ms = medir_por_item(FEATURE_CONFIGS_V2[feature_config_name], amostra)

    X_tr, X_va = np.load(f_tr), np.load(f_va)
    (model, scaler), treino_s = medir_s(lambda: train_svm(X_tr, y_train, kernel='rbf', C=1.0))
    _, pred_s = medir_s(lambda: model.predict(scaler.transform(X_va)))
    predicao_ms = pred_s / len(X_va) * 1000

    gravar_tempo(TIMING_V2_PATH, exp_id, 'validacao', len(X_va), extracao_ms=extracao_ms,
                 treino_s=treino_s, predicao_ms=predicao_ms,
                 hardware=hardware_info(usado='CPU (LBP + SVM)'))
    print(f"[{exp_id}] extração {extracao_ms:.2f} ms/img | treino {treino_s:.1f} s | predição {predicao_ms:.3f} ms/img")


for _exp_id, _cfg in CONFIGS_V2.items():
    medir_tempo_validacao(_exp_id, _cfg)

display(carregar_tempos(TIMING_V2_PATH))


## Escolher a melhor configuração e avaliar UMA vez no teste

Só a configuração vencedora vai para o teste, uma única vez. Escolha em `BEST_EXP_ID` depois de olhar a comparação acima.

In [ ]:
# Apenas a configuração escolhida na validação vai para o teste

BEST_EXP_ID = 'lbp_svm_v2_02_grid'  # <-- alterar conforme a comparação acima

final_id = BEST_EXP_ID + '_FINAL_TEST'
if ja_registrado(final_id):
    raise RuntimeError(f"O teste de '{BEST_EXP_ID}' já foi registrado no log do v2. Ele é avaliado uma única vez.")

feature_config_name = CONFIGS_V2[BEST_EXP_ID]
exp_dir = os.path.join(RUNS_DIR, BEST_EXP_ID)

# Retreina com as características salvas do treino (mesmo seed), sem depender de variáveis em memória
X_train = np.load(os.path.join(exp_dir, 'X_train.npy'))
model, scaler = train_svm(X_train, y_train, kernel='rbf', C=1.0)

X_test = get_features(exp_dir, 'test', test_paths, feature_config_name)
_, t_pred_teste_s = medir_s(lambda: model.predict(scaler.transform(X_test)))   # tempo de predição no teste

test_results = evaluate(model, scaler, X_test, y_test)

print(f"RESULTADO FINAL NO TESTE ({BEST_EXP_ID})")
print(f"HTER: {test_results['HTER']*100:.2f}%")
print(f"AUC:  {test_results['AUC']:.4f}")
print(f"EER:  {test_results['EER']*100:.2f}%  (usa o melhor limiar do próprio teste, então é otimista)")

In [ ]:
# Registra o resultado final de teste no log do v2

log_experiment_v2(
    exp_id=final_id,
    metodo='LBP+SVM (v2 224)',
    feature_config=feature_config_name,
    modelo_config='kernel=rbf, C=1.0; escolhida pela validação',
    hter_test=test_results['HTER'],
    auc_test=test_results['AUC'],
    eer_test=test_results['EER'],
    obs='Métrica final de teste do LBP+SVM no v2',
)

# tempo de predição do teste (timing_v2.csv)
gravar_tempo_teste(TIMING_V2_PATH, BEST_EXP_ID, len(X_test), t_pred_teste_s,
                   hardware=hardware_info(usado='CPU (LBP + SVM)'))


In [ ]:
# Salva as predições do teste para análise posterior (não sobrescreve)

predictions_path = os.path.join(exp_dir, 'predictions_test.csv')
if os.path.exists(predictions_path):
    print(f"{predictions_path} já existe; não sobrescrevi.")
else:
    pd.DataFrame({
        'image_path': test_paths,
        'label_true': y_test,
        'label_pred': test_results['y_pred'],
        'score': test_results['y_scores'],
    }).to_csv(predictions_path, index=False)
    print(f"Predições salvas em {predictions_path}")